In [1]:
import numpy as np
import pandas as pd
from src.segment import find_corners, build_zones
import plotly.graph_objects as go
from src.loader import load_centerline
from src.score import compute_scores_s1, compute_scores_s2, compute_scores_s3, plot_best_score

In [2]:
cl = load_centerline()

In [3]:
cl_, apexes = find_corners(cl)
cl_, zones = build_zones(cl_, apexes)

In [4]:
# im going to create multiple segmentations and see how they score.
# first i will create an array of multiple values for the parameters
n = 10000
l = round(cl.attrs["length"]) + 1
abs_max = cl["curvature"].abs().max()
np.random.seed(42)
## |curvature| below this = straight-ish
k_straight = np.random.uniform(low=0, high=abs_max, size=n)
# straight-ish stretch must be at least this long
min_straight_m = np.random.randint(low=1, high=50, size=n, dtype=int)
# |curvature| above this = clearly a corner
k_corner = np.random.uniform(low=0, high=abs_max, size=n)
# rolling-mean window (odd number, 1 point = 1 m)
low = 1
high = 50
odd_numbers = np.arange(low if low % 2 else low + 1, high, 2)
smooth_window_m = np.random.choice(odd_numbers, size=n)
# curvature between two apexes must stay above this to be one corner
k_link = np.random.uniform(low=0, high=abs_max,  size=n)
# and the apexes must be at most this far apart
max_link_m = np.random.randint(low=1, high=80, size=n, dtype=int)

In [5]:
space = pd.DataFrame(
    data= 
    {
        "k_straight": k_straight,
        "min_straight_m": min_straight_m,
        "k_corner": k_corner,
        "smooth_window_m": smooth_window_m,
        "k_link": k_link,
        "max_link_m": max_link_m
    }
)
space

,k_straight,min_straight_m,k_corner,smooth_window_m,k_link,max_link_m
0,0.132887,32,0.196267,33,0.075165,56
1,0.337314,36,0.334081,37,0.009656,20
2,0.259712,37,0.326824,35,0.344370,64
3,0.212404,48,0.313683,9,0.165795,17
4,0.055356,14,0.034769,29,0.057392,16
...,...,...,...,...,...,...
9995,0.304297,4,0.097588,45,0.230912,1
9996,0.318437,7,0.210063,1,0.264321,43
9997,0.335893,12,0.009108,47,0.195260,69
9998,0.141029,13,0.087361,13,0.298954,75


In [6]:
space1 = space.copy()
space2 = space.copy()
space3 = space.copy()

## Score Func1: yhat = y

In [7]:
labels_s1 = cl_[["s", "is_straight"]]
# straight = 0, corner = 1
labels_s1['type'] = np.where(labels_s1['is_straight'], 0, 1)
labels_s1 = labels_s1[["s", "type"]]

In [8]:
scores1 = compute_scores_s1(n = n, space= space1, cl = cl, labels = labels_s1)

In [9]:
scores1

,k_straight,min_straight_m,k_corner,smooth_window_m,k_link,max_link_m,score
0,0.020243,20,0.288647,3,0.069456,6,0.992556
1,0.021930,23,0.247821,3,0.180329,35,0.992556
2,0.022881,14,0.057151,1,0.186170,51,0.975186
3,0.018338,20,0.303967,3,0.045667,37,0.968983
4,0.019099,24,0.194603,5,0.306749,24,0.965261
5,0.019123,23,0.117206,5,0.303230,30,0.965261
6,0.023581,28,0.336669,1,0.229130,40,0.964020
7,0.012779,13,0.018278,7,0.023764,17,0.959057
8,0.015313,13,0.075458,3,0.222975,79,0.957816
9,0.026756,17,0.081851,3,0.021356,3,0.947891


In [10]:
plot_best_score(space_scores = scores1, cl= cl)

space:
 k_straight          0.020243
min_straight_m     20.000000
k_corner            0.288647
smooth_window_m     3.000000
k_link              0.069456
max_link_m          6.000000
score               0.992556
Name: 0, dtype: float64


## Score Func2: yhat=y AND zone_hat = zone

In [11]:
labeles_s2 = cl_[["s", "is_straight", "zone"]]
labeles_s2['type'] = np.where(labeles_s2['is_straight'], 0, 1)
labeles_s2 = labeles_s2[["s", "type", "zone"]]
labeles_s2 

,s,type,zone
0,0.0,0,1
1,1.0,0,1
2,2.0,0,1
3,3.0,0,1
4,4.0,0,1
...,...,...,...
801,801.0,1,32
802,802.0,0,1
803,803.0,0,1
804,804.0,0,1


In [12]:
scores2 = compute_scores_s2(n = n, space=space2, cl=cl, labels=labeles_s2)

In [13]:
scores2

,k_straight,min_straight_m,k_corner,smooth_window_m,k_link,max_link_m,score
0,0.028755,37,0.015285,13,0.275325,28,0.653846
1,0.037819,20,0.031860,9,0.062890,3,0.651365
2,0.019775,1,0.105757,7,0.303187,61,0.353598
3,0.031698,20,0.006631,7,0.052336,2,0.307692
4,0.024130,17,0.024384,17,0.031873,70,0.289082
5,0.028407,7,0.353870,3,0.037633,17,0.277916
6,0.016697,5,0.021947,21,0.073110,40,0.277916
7,0.023762,1,0.304506,19,0.057138,25,0.277916
8,0.033726,6,0.192037,1,0.098146,21,0.275434
9,0.030764,19,0.018035,15,0.314225,65,0.271712


In [14]:
plot_best_score(space_scores=scores2, cl = cl)

space:
 k_straight          0.028755
min_straight_m     37.000000
k_corner            0.015285
smooth_window_m    13.000000
k_link              0.275325
max_link_m         28.000000
score               0.653846
Name: 0, dtype: float64


## score func3: the overlap regions

In [17]:
labels_s3 = cl_[["s", "is_straight", "zone"]]
labels_s3['type'] = np.where(labels_s3['is_straight'], 0, 1)
labels_s3 = labels_s3[["s", "type", "zone"]]
labels_s3

,s,type,zone
0,0.0,0,1
1,1.0,0,1
2,2.0,0,1
3,3.0,0,1
4,4.0,0,1
...,...,...,...
801,801.0,1,32
802,802.0,0,1
803,803.0,0,1
804,804.0,0,1


In [20]:
scores3 = compute_scores_s3(n = n, space = space3, cl = cl, labels = labels_s3)

In [21]:
plot_best_score(space_scores=scores3, cl=cl)

space:
 k_straight          0.018391
min_straight_m     29.000000
k_corner            0.034232
smooth_window_m     7.000000
k_link              0.288861
max_link_m         73.000000
score               0.885856
Name: 0, dtype: float64
